In [1]:
from ingest import load_lessons_data
from minsearch import Index
from dotenv import load_dotenv
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
import os


groq_base_url = "https://api.groq.com/openai/v1"
load_dotenv()

model = "groq/compound-mini"

INSTRUCTIONS = """
Your task is to answer questions from the lessons
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=groq_base_url
)

def build_context(search_results):
    chunks = []

    for doc in search_results:
        chunks.append(
            f"""FILE: {doc['filename']}{doc['content']}"""
        )

    return "\n\n".join(chunks)

def llm(prompt):
    input_messages = [
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=input_messages
    )
    answer = response.output_text
    usage = response.usage
    return answer, usage


## Q1. How many lesson pages

How many lesson pages are in the dataset?

In [2]:
documents = load_lessons_data()
print(f"There are {len(documents)} lesson pages in llm-zoomcamp")

There are 72 lesson pages in llm-zoomcamp


## Q2. Indexing and searching

Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

How does the agentic loop keep calling the model until it stops?

### What's the filename of the first result?

In [3]:
query = "How does the agentic loop keep calling the model until it stops?"

index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)
index.fit(documents)

search_results = index.search(query, num_results=2)
search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

## Q3. RAG

Build a RAG over the index from Q2 and answer the query:

How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

In [4]:
query = "How does the agentic loop keep calling the model until it stops?"

context = build_context(search_results)
prompt = PROMPT_TEMPLATE.format(question=query, context=context)
answer, usage = llm(prompt)
print(usage)

ResponseUsage(input_tokens=7701, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=433, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=8134)


## Q4. Chunking

How many chunks do you get?

In [5]:
chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)
len(chunks)

295

## Q5. RAG with chunking

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

chunk_index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)
chunk_index.fit(chunks)
search_results = chunk_index.search(query, num_results=5)
context = build_context(search_results)
prompt = PROMPT_TEMPLATE.format(question=query, context=context)
answer, usage = llm(prompt)
print(usage)

ResponseUsage(input_tokens=4996, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=522, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=5518)


## Q6. Turning it into an agent

To get the output for this question, please run **framework.py* and use the below command

**python framework.py**

Enter the question in command prompt, so finally you can get the answer with search call in the console. Here we used pydantic ai package.